# Descubrimiento de Contraseñas mediante un Algoritmo Genético con Metaheurísticas


El objetivo será descubrir una contraseña secreta previamente definida mediante el uso de un algoritmo genético, que evolucione la población de posibles contraseñas hasta encontrar la que coincida exactamente con la contraseña objetivo.

La calidad de las contraseñas generadas se medirá mediante una función de aptitud que evaluará la similitud con la contraseña objetivo según los siguientes factores:

**Coincidencia exacta de caracteres en la posición correcta:**
- 1 punto positivo por cada carácter correcto en la posición exacta.

**Coincidencia parcial de caracteres en posiciones incorrectas:**
- 0.5 puntos positivos por cada carácter correcto en una posición diferente.

El algoritmo buscará una contraseña que consiga una puntuación igual a la longitud de la contraseña objetivo, lo que indicará que ha sido encontrada con exactitud. No hay una longitud previamente definida, esta será determinada por la longitud de la contraseña objetivo y podrá variar.

Cuando se encuentre una contraseña con la puntuación máxima, **el algoritmo finalizará inmediatamente** y indicará la generación en la que se ha encontrado.

Si no se encuentra la contraseña exacta en el número máximo de generaciones permitido, **el algoritmo finalizará mostrando la mejor aproximación encontrada hasta el momento**, junto con su puntuación de aptitud.


In [ ]:
import random

In [ ]:
TAMAÑO_POBLACION = 10  # Número de individuos por generación
MAX_GENERACIONES = 1000  # Número máximo de generaciones por evolución

# Definimos la contraseña objetivo
#CONTRASENA_OBJETIVO = "Hola1234"
CONTRASENA_OBJETIVO = "Mireia@Consarnau!"
LONGITUD_CONTRASENA = len(CONTRASENA_OBJETIVO)

# OBJETIVO ahora es la puntuación máxima alcanzable
OBJETIVO = LONGITUD_CONTRASENA  # 1 punto por carácter exacto + 0.5 por carácter en posición incorrecta

# Lista de caracteres válidos para generar las posibles contraseñas
GENES = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789!@#$%^&*"

In [ ]:
# Generar un cromosoma (lista de caracteres) con la misma longitud que la contraseña objetivo.
# Los caracteres son seleccionados aleatoriamente de la variable GENS.

import random

def crear_cromosoma():
    return [random.choice(GENES) for _ in range(LONGITUD_CONTRASENA)]

In [ ]:
# Aplicar mutaciones a un cromosoma según una tasa de mutación.
# Para cada gen del cromosoma, si un número aleatorio es menor que la tasa de mutación,
# se reemplaza el gen por otro carácter aleatorio de GENS.

def mutar(cromosoma, tasa_mutacion):
    cromosoma_mutado = []
    for gen in cromosoma:
        if random.uniform(0, 1) < tasa_mutacion:
            cromosoma_mutado.append(random.choice(GENES))
        else:
            cromosoma_mutado.append(gen)
    return cromosoma_mutado

In [ ]:
# Generar un nuevo cromosoma (hijo) a partir de dos padres (par1 y par2).
# Seleccionar un punto de corte aleatorio y combinar los genes del primer padre antes del punto de corte,
# y del segundo padre después del punto de corte. Aplicar mutación al hijo con una tasa del 10%.

def aparear(par1, par2):
    punto_corte = random.randint(0, LONGITUD_CONTRASENA - 1)
    hijo = par1[:punto_corte] + par2[punto_corte:]
    return mutar(hijo, 0.1)

In [ ]:
""" Calcula la aptitud de un cromosoma comparándolo con la contraseña objetivo.
La puntuación se calcula según dos criterios:
1. +1 punto por cada carácter correcto en la posición exacta.
2. +0.5 puntos por cada carácter correcto en una posición incorrecta (sin duplicar los ya contados en la posición correcta).
"""
def calcular_aptitud(cromosoma, contrasena_objetivo):
    aptitud = 0
    restantes_obj = []  # Caracteres de la contraseña objetivo que no han coincidido exactamente
    restantes_crom = []  # Caracteres del cromosoma que no han coincidido exactamente

    # **1er paso: Sumar +1 por cada coincidencia exacta**
    for i in range(len(contrasena_objetivo)):
        if cromosoma[i] == contrasena_objetivo[i]:
            aptitud += 1
        else:
            restantes_obj.append(contrasena_objetivo[i])  # Guardamos los no correctos
            restantes_crom.append(cromosoma[i])

    # **2do paso: Sumar +0.5 por coincidencias en posición incorrecta**
    for char in restantes_crom:
        if char in restantes_obj:
            aptitud += 0.5
            restantes_obj.remove(char)  # Lo eliminamos para evitar recuento

    return aptitud

In [ ]:
generacion = 1
poblacion = []
mejor_solucion = None  # Guardará la mejor aproximación encontrada

# Crear la población inicial con cromosomas aleatorios y calcular su aptitud
for _ in range(TAMAÑO_POBLACION):
    cromosoma = crear_cromosoma()
    poblacion.append((cromosoma, calcular_aptitud(cromosoma, CONTRASENA_OBJETIVO)))

print("Iniciamos el proceso de apareamiento:")

while generacion <= MAX_GENERACIONES:
    # Ordenar la población según su aptitud (de mayor a menor)
    poblacion = sorted(poblacion, key=lambda x: x[1], reverse=True)

    # Guardamos la mejor solución hasta ahora
    mejor_solucion = poblacion[0]

    # Comprobamos si ya se ha encontrado la contraseña exacta
    if mejor_solucion[1] == OBJETIVO:
        print(f"Contraseña encontrada en la generación {generacion}: {''.join(mejor_solucion[0])}")
        break  # Salimos inmediatamente del bucle

    nueva_generacion = []

    # Aparear y crear nuevos hijos (90% del tamaño de la población)
    for _ in range(int(0.9 * TAMAÑO_POBLACION)):
        padre1 = random.choice(poblacion[:TAMAÑO_POBLACION // 2])  # Selección de los mejores
        padre2 = random.choice(poblacion[:TAMAÑO_POBLACION // 2])
        hijo = aparear(padre1[0], padre2[0])
        aptitud_hijo = calcular_aptitud(hijo, CONTRASENA_OBJETIVO)

        # Comprobamos si el nuevo hijo ha encontrado la contraseña exacta
        if aptitud_hijo == OBJETIVO:
            print(f"Contraseña encontrada en la generación {generacion}: {''.join(hijo)}")
            exit()  # Salimos inmediatamente del programa

        nueva_generacion.append((hijo, aptitud_hijo))

    # Mantenemos un 10% de la mejor población anterior para estabilizar la búsqueda
    poblacion = nueva_generacion + poblacion[:int(0.1 * TAMAÑO_POBLACION)]

    print(f"Generación {generacion}: Mejor cadena: {''.join(mejor_solucion[0])} Aptitud: {mejor_solucion[1]}")

    generacion += 1

# Si no se ha encontrado la contraseña, mostrar la mejor aproximación
if mejor_solucion[1] < OBJETIVO:
    print(f"\nNo se ha encontrado la contraseña exacta. Mejor aproximación encontrada: {''.join(mejor_solucion[0])} con aptitud {mejor_solucion[1]}")


Iniciamos el proceso de apareamiento:
Generación 1: Mejor cadena: hkJdfIBtoysi4nhOp Aptitud: 3.5
Generación 2: Mejor cadena: 8&S1JRBtoysi4nhOp Aptitud: 3.5
Generación 3: Mejor cadena: M3uEWyO!Rtsi4nhOp Aptitud: 4.5
Generación 4: Mejor cadena: M3uCWyO!Rtsi4nhzE Aptitud: 5.0
Generación 5: Mejor cadena: M3uCWyO!Rtsi4nhzE Aptitud: 5.0
Generación 6: Mejor cadena: M3uEWyO!5tsinCWnh Aptitud: 5.0
Generación 7: Mejor cadena: MkJ6fIxtoysinn@OA Aptitud: 5.5
Generación 8: Mejor cadena: M3uEWyO!5tsinn@0E Aptitud: 5.5
Generación 9: Mejor cadena: M3xEWyC!5tsinn@0E Aptitud: 5.5
Generación 10: Mejor cadena: M3uEsFe!*psinn@0E Aptitud: 6.0
Generación 11: Mejor cadena: M3uEsFe!*psinn@0E Aptitud: 6.0
Generación 12: Mejor cadena: M3uEsFe!*psinn@0E Aptitud: 6.0
Generación 13: Mejor cadena: M3uEsFe!5^sinn@Fh Aptitud: 6.0
Generación 14: Mejor cadena: M3x9WyC!5^sinn@ut Aptitud: 6.5
Generación 15: Mejor cadena: M3x9WyC!5^sinn@ut Aptitud: 6.5
Generación 16: Mejor cadena: M3x9WyC!5^sinn@ut Aptitud: 6.5
Generación 

# Conclusiones
Este ejercicio ha sido diseñado con un tamaño de población de 10, lo que permite que el algoritmo funcione de manera rápida y eficiente en un espacio pequeño de soluciones.

Con esta configuración, el algoritmo puede detenerse inmediatamente cuando encuentra la contraseña objetivo, optimizando el tiempo de ejecución y evitando iteraciones innecesarias. Si la solución no se encuentra dentro del límite de generaciones, se retorna la mejor aproximación encontrada hasta el momento.

Si la contraseña no se encuentra dentro del límite de generaciones, el programa muestra la mejor aproximación encontrada, asegurando que siempre obtengamos un resultado útil.

Para hacerlo más efectivo en casos más complejos, se podrían aplicar algunas mejoras, como aumentar el tamaño de la población para explorar más opciones, añadir más mutación para evitar quedar atrapados en una solución subóptima, o seleccionar individuos más diversos para mejorar la búsqueda.